In [ ]:
# MWE to showcase the use of the SpinSquaredProjectorBlock to retrieve symmetry-adapted states from a reference circuit

from scipy.linalg import eigh
import numpy as np
from pyscf import gto, scf
from openfermion import hamiltonians

from qarp.operators import JordanWigner
from qarp.operators.ucc import ucc_singles_and_doubles
from qarp.blocks import MappedONVStateBlock, TrotterAnsatzBlock, CompositeBlock
from qarp.algorithms import VQE
from qarp.algorithms import StateVector
from qarp.optimizers import ScipyOptimizer
from qarp.engines import QarpEngine
from qarp.operators.integrals import restricted_integrals_to_fermion_operator
from qarp.operators.compat import from_openfermion

from qarp.endianness import bits_to_label, label_to_bits
from qarp.plotting import plot
import qarpx as qx

from qarp.blocks import SpinSquaredProjectorBlock
from qarp.operators import FullyCommuting
from qarp.operators.pyscf import active_space_from_mf



In [ ]:
# Run VQE calculation for N2 molecule in minimal basis set to get a spin-broken reference state 
bl = 1.
geometry = f"N 0 0 0; N 0 0  {bl}"
mol = gto.M(atom=geometry, basis="sto3g", verbose=-1, symmetry=True)
mol.build()
mf = scf.RHF(mol)
mf.kernel()
mol.build()

integrals, onv = active_space_from_mf(mf, 2, 3)

fermion_operator = restricted_integrals_to_fermion_operator(*integrals)

qop = JordanWigner().encode_operator(fermion_operator)

fucc, symbols = ucc_singles_and_doubles(onv, spin_conserving=True, generalised=True)
qucc = JordanWigner().encode_operator(fucc)

blocks = [MappedONVStateBlock(onv, JordanWigner()),
        TrotterAnsatzBlock(
            len(onv),
            qucc,
            symbols,
            steps=1,
            time=1,
            order=1,
            grouping=FullyCommuting(),
            imaginary=True
        )
    ]
ket = CompositeBlock(blocks).build()

# VQE
initial_parameters = np.random.random(len(ket.symbols))
options = {"maxiter": 80}
optimizer = ScipyOptimizer(method="COBYLA", options=options)
vqe = VQE(
        operator=qop,
        ket=ket,
        primitive=StateVector(),
        engine=QarpEngine(),
        initial_parameters=initial_parameters,
        optimizer=optimizer,
        verbose=False,
).build()

e_vqe, x_vqe = vqe.run()
print("VQE Energy: ", e_vqe)

ham_mat = qop.sparse_matrix().toarray()  # qarpx LSB — same basis as every statevector below

eigenvalues, eigenvectors = eigh(ham_mat)
n2_gs = eigenvectors[:, np.argmin(eigenvalues)]
n2_gse = eigenvalues[np.argmin(eigenvalues)]

# Overlap of VQE with exact GS (sparse_matrix and the qarpx statevector share the LSB basis)
params_vqe = vqe.optimal_parameters  # result.x mapped onto the sorted symbol tuple (§17)
n2_wfn_vqe = ket.set_symbols(params_vqe)
n2_wfn_vqe_SV = np.array(qx.QarpSimulator().statevector(n2_wfn_vqe.flatten(), n2_wfn_vqe.n_qubits))

print("Overlap of VQE with exact GS: ", n2_wfn_vqe_SV @ n2_gs)
print("Exact GS Energy: ", n2_gse)

In [ ]:
# We see here that the VQE state has a broken spin symmetry by computing the variance of the S^2 operator 
# We also see that the ground state of N2 is a single state (S=0) and the VQE state is a mixture

# openfermion supplies the S² model; convert at the boundary and realize it in qarpx LSB.
s2 = from_openfermion(hamiltonians.s_squared_operator(len(onv) // 2))
s2mat = s2.sparse_matrix(len(onv)).toarray()

var_gs  = n2_gs.T.conj() @ s2mat @ s2mat @ n2_gs - (n2_gs.conj() @ s2mat @ n2_gs)**2
var_vqe = n2_wfn_vqe_SV.T.conj() @ s2mat @ s2mat @ n2_wfn_vqe_SV - (n2_wfn_vqe_SV.conj() @ s2mat @ n2_wfn_vqe_SV)**2

print("Var(S^2) for exact GS and VQE: ", var_gs, var_vqe)
print("<S^2> for exact GS and VQE: ", n2_gs.T.conj() @ s2mat @ n2_gs, n2_wfn_vqe_SV.T.conj() @ s2mat @ n2_wfn_vqe_SV)

In [ ]:
# Let's apply the projector on top of the VQE circuit to retrieve the symmetry-adapted state

def _statevector(block):
    block.build()
    return np.array(qx.QarpSimulator().statevector(block.flatten(), block.n_qubits))

def _ancilla_zero_branch(statevector: np.ndarray, n_anc: int, n_target: int) -> np.ndarray:
    """Extract target amplitudes with ancillas fixed to |0...0>."""
    indices = np.array([target_state << n_anc for target_state in range(2**n_target)])
    return statevector[indices]

# We will project to S=0 and thus Ms=0 (the only possible value for S=0)
n_qubits = len(onv)
S = 0
Ms = 0

# Projector block
projector = SpinSquaredProjectorBlock(n_qubits=n_qubits, S=S, Ms=Ms)

# VQE block
vqe_block = vqe.final_block
vqe_block.name = "VQE"
vqe_block.target_qubits = projector.system_qubits

# Build the composite circuit and retrieve the statevector of the projected state
spin_projected_circuit = CompositeBlock(
    [vqe_block, projector],
    n_qubits=projector.n_qubits,
).build()

# Retrieve the statevector of the projected state and the branch corresponding to the system qubits (ancillas are used in SpinSquaredProjectorBlock)
state = _statevector(spin_projected_circuit)
target_branch = _ancilla_zero_branch(state, projector.num_controls, n_qubits)

# The projected state is not normalized, we need to normalize it
target_branch /= np.linalg.norm(target_branch)


# Finally, compute the expectation value and variance of S^2 with the projected state
var_proj = target_branch.T.conj() @ s2mat @ s2mat @ target_branch - (target_branch.T.conj() @ s2mat @ target_branch)**2
exp_proj = target_branch.T.conj() @ s2mat @ target_branch

print("Var(S^2) for projected state (should be zero): ", var_proj)
print("<S^2> for projected state (should be S(S+1)): ", exp_proj)


In [ ]:
# Plot the VQE + spin projector circuit. See how the projector spans now over some ancilla qubits

plot(spin_projected_circuit)

In [ ]:
# Compute the energy of this projected state and its overlap with the exact ground state

en_projected = target_branch.T.conj() @ ham_mat @ target_branch
ov_projected = target_branch.T.conj() @ n2_gs

print("Overlap of projected state with exact GS and projected energy: ", ov_projected, en_projected)